In [1]:
import json
from src.db import DbConnection
from src.utils.gembase_utils import GembaseUtils
from src.runtime_constants import rr

In [2]:
#conn = DbConnection()
conn = DbConnection(rr.ENV_PROD, True)

In [4]:
conn.close()

In [3]:
tags1 = [
    {"from": ["behaviors__complete", "behaviors__collect"], "to": "behaviors__utilize"},
    {"from": ["behaviors__empower"], "to": "behaviors__cultivate"},
    {"from": ["behaviors__solve"], "to": "behaviors__improvise"},
    {"from": ["behaviors__immerse"], "to": "behaviors__experience"},
    {"from": ["behaviors__cooperate"], "to": "behaviors__bond"},
    {"from": ["behaviors__compete"], "to": "behaviors__challenge"},
    {"from": ["behaviors__chill", "behaviors__react"], "to": ""},
]

tags2 = [
    {"from": ["behaviors__experience"], "to": "behaviors__immerse"},
]

date = "13_11_23"

all_from = []
for it in tags2:
    all_from += it["from"]
all_from_db = GembaseUtils.values_arr_to_db_in(all_from)

In [ ]:
conn.query(f"""
CREATE TABLE archive.tags_rename_{date}
SELECT * FROM tagged_data.tags t
 WHERE t.tag_id_int IN (
 SELECT m.id FROM app.map_tag_id m WHERE m.tag_id IN ({all_from_db})
)
""")

for it in tags:
    if it["to"] != "":
        tags_from_db = GembaseUtils.values_arr_to_db_in(it["from"])
        conn.query(f"""
        INSERT INTO tagged_data.tags (app_id_int, tag_id_int, tag_rank)
        SELECT z.app_id_int, z.tag_id_int, IF(z.tag_rank = 999, 0, z.tag_rank) AS tag_rank FROM (
        SELECT t.app_id_int, m_n.id AS tag_id_int, MIN(IF(t.tag_rank = 0, 999, t.tag_rank)) AS tag_rank
          FROM tagged_data.tags t,
               app.map_tag_id m_o,
               app.map_tag_id m_n
         WHERE t.tag_id_int = m_o.id
           AND m_o.tag_id IN ({tags_from_db})
            AND m_n.tag_id = {it["to"]}
            GROUP BY t.app_id_int, m_n.id ) z
        """)


conn.query(f"""
DELETE FROM tagged_data.tags t
 WHERE t.tag_id_int IN (
 SELECT m.id FROM app.map_tag_id m WHERE m.tag_id IN ({all_from_db})
)
""")

In [4]:
conn.query(f"""
CREATE TABLE archive.survey_tags_rename_{date}
SELECT * FROM survey_v2.survey_instance
""")

limit = 100
page = 0
while True:
    print(f"Page {page}")

    rows = conn.select_all(f"""
    SELECT id, client_data, server_data
      FROM survey_v2.survey_instance
      order by id
      limit {page * limit}, {limit}
    """)

    if len(rows) == 0:
        break

    page += 1

    bulk_data = []
    for row in rows:
        client_data: str = row["client_data"]
        server_data: str = row["server_data"]

        for it in tags2:
            tag_to = it["to"]
            for tag_from in it["from"]:
                if client_data is not None:
                    client_data = client_data.replace(tag_from, tag_to)
                if server_data is not None:
                    server_data = server_data.replace(tag_from, tag_to)

        bulk_data.append((client_data, server_data, row["id"]))

    conn.bulk("""
    UPDATE survey_v2.survey_instance
       SET client_data = %s, server_data = %s
     WHERE id = %s
    """, bulk_data)
    conn.commit()


Page 0
Page 1
Page 2
Page 3
Page 4
Page 5
Page 6
Page 7
Page 8
Page 9
Page 10
Page 11
Page 12
Page 13
Page 14
Page 15
Page 16
Page 17
Page 18
Page 19
Page 20
Page 21
Page 22
Page 23
Page 24
Page 25
Page 26
Page 27
Page 28
Page 29
Page 30
Page 31
Page 32
Page 33
Page 34
Page 35
Page 36
Page 37


In [14]:
# conn.query(f"""
# CREATE TABLE archive.tags_rename_survey_{date}
# SELECT * FROM survey_data.survey_tags t
#  WHERE t.tag_id IN ({all_from_db})
# """)

for it in tags:
    if it["to"] != "":
        tags_from_db = GembaseUtils.values_arr_to_db_in(it["from"])
        conn.query(f"""
        INSERT INTO survey_data.survey_tags (survey_meta_id, survey_instance, tag_id, tag_value)
        SELECT z.survey_meta_id, z.survey_instance, z.tag_id, z.tag_value FROM (
        SELECT t.survey_meta_id, t.survey_instance, '{it["to"]}' as tag_id, AVG(t.tag_value) as tag_value
          FROM survey_data.survey_tags t
         WHERE t.tag_id IN ({tags_from_db})
         GROUP BY t.survey_meta_id, t.survey_instance ) z
        """)


conn.query(f"""
DELETE FROM survey_data.survey_tags t
 WHERE t.tag_id IN ({all_from_db})
""")

In [4]:
from src.server.models.platform_values.platform_values_utils import PlatformValuesUtils

PlatformValuesUtils.recreate_platform_values_tables(conn)

In [5]:
conn.commit()
conn.close()